# Sprint 2: Frozen Features And Single-Backbone MLPs

Thin Colab runner around the repository scripts. Keep implementation in `src/dl_midterm/` and `scripts/`.

Run `00_colab_setup.ipynb` first so the HAM10000 Drive bundle is extracted into the repo root.

In [ ]:
from google.colab import drive
from pathlib import Path
import os
import subprocess

REPO_URL = 'https://github.com/KasimDeliaci/dl-midterm-project.git'
REPO_ROOT = Path('/content/dl-assignment')
BUNDLE_CANDIDATES = [
    Path('/content/drive/MyDrive/ham10000_colab_bundle.tar'),
    Path('/content/drive/MyDrive/Colab Notebooks/ham10000_colab_bundle.tar'),
    Path('/content/drive/MyDrive/dl-assignment/ham10000_colab_bundle.tar'),
]
DRIVE_ARTIFACTS = Path('/content/drive/MyDrive/dl-midterm-artifacts')

drive.mount('/content/drive', force_remount=True)
DRIVE_BUNDLE = next((path for path in BUNDLE_CANDIDATES if path.exists()), BUNDLE_CANDIDATES[0])

if not (REPO_ROOT / 'pyproject.toml').exists():
    subprocess.run(['git', 'clone', REPO_URL, str(REPO_ROOT)], check=True)
else:
    subprocess.run(['git', '-C', str(REPO_ROOT), 'pull', '--ff-only'], check=True)

os.chdir(REPO_ROOT)
print('Repo root:', Path.cwd())
print('Drive bundle exists:', DRIVE_BUNDLE.exists(), DRIVE_BUNDLE)
print('Checked bundle candidates:')
for path in BUNDLE_CANDIDATES:
    print(' -', path, path.exists())

In [ ]:
required_paths = [
    REPO_ROOT / 'data/raw/ham10000',
    REPO_ROOT / 'data/metadata/HAM10000_metadata.csv',
    REPO_ROOT / 'data/splits/train.csv',
    REPO_ROOT / 'data/splits/val.csv',
    REPO_ROOT / 'data/splits/test.csv',
]

if not all(path.exists() for path in required_paths):
    if not DRIVE_BUNDLE.exists():
        raise FileNotFoundError(f'Missing Drive bundle: {DRIVE_BUNDLE}')
    subprocess.run(['tar', '--warning=no-unknown-keyword', '-xf', str(DRIVE_BUNDLE), '-C', str(REPO_ROOT)], check=True)

missing = [str(path.relative_to(REPO_ROOT)) for path in required_paths if not path.exists()]
if missing:
    raise RuntimeError(f'Missing required dataset paths after extraction: {missing}')

print('Dataset and preserved Sprint 1 split CSVs are available.')
for path in required_paths:
    print(path.relative_to(REPO_ROOT))

Install dependencies after cloning or updating the repository.

In [ ]:
!python -m pip install -q uv
!uv sync

Extract frozen ImageNet-pretrained features for ResNet50, MobileNetV2, and EfficientNetB0. In Colab, keep `artifacts/features` on Drive if you want caches to persist between sessions.

In [ ]:
%%bash
uv run python scripts/extract_features.py \
  --config configs/dataset/selected_dataset.yaml \
  --default-config configs/default.yaml \
  --source frozen \
  --batch-size 64

Train the three single-backbone MLP baselines and export report-ready summary assets.

In [ ]:
%%bash
uv run python scripts/train_mlp.py \
  --config configs/experiments/frozen_feature_matrix.yaml \
  --default-config configs/default.yaml \
  --dataset-config configs/dataset/selected_dataset.yaml \
  --feature-source frozen \
  --fusion none

If runs already exist and you only need refreshed aggregate tables/plots, rerun evaluation.

In [ ]:
!uv run python scripts/evaluate_runs.py --feature-source frozen

Sync report-ready artifacts and frozen feature caches back to Drive.

In [ ]:
from pathlib import Path
import shutil

DRIVE_ARTIFACTS = Path('/content/drive/MyDrive/dl-midterm-artifacts')
paths_to_sync = [
    Path('artifacts/features/ham10000/frozen'),
    Path('artifacts/report_assets/tables'),
    Path('artifacts/report_assets/figures'),
]

for src in paths_to_sync:
    if src.exists():
        dst = DRIVE_ARTIFACTS / src
        dst.parent.mkdir(parents=True, exist_ok=True)
        shutil.copytree(src, dst, dirs_exist_ok=True)
        print('Synced', src, '->', dst)